# 04 — Multi-class Classification (BiGRU + MLP-FV)

## Summary

**Purpose:** Test whether the dataset size — not the model architecture — is the
binding constraint in binary PLA.

BiGRU + SpecAugment (on DGT matrices) and MLP-FV (on preprocessed feature vectors) are both trained on all 12 devices (1340 windowed DGT
samples, ~112 per device) for 12-class identification. Results are compared against
the binary LOTO ADR from notebook 02.

**Hypothesis:** If 12-class accuracy > binary LOTO ADR, the per-classifier sample
starvation (4 devices, ~50 windowed samples per binary model) is the bottleneck,
not the model.

**Design choices:**
- Group-aware 70/15/15 split: all windows of one transient stay in the same partition
- Z-score normalisation fit on train only
- 5 seeds → different splits + weight initialisations → mean ± std
- Inverse-frequency class weights correct for device imbalance (device10–12: 60 windows vs device2: 140)
- Random baseline: 1/12 ≈ 8.3%

**Prerequisites:** `00_setup.ipynb` (builds `dgt_windowed.h5`).

## 1. Setup

In [ ]:
import os, sys
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import f1_score, confusion_matrix, accuracy_score

ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from binary_pla.config import (
    SEEDS, BATCH_SIZE, NB04_ID, VAL_FRAC, TEST_FRAC,
    N_DEVICES, N_WIN, DEVICE_NAMES, NB04_DL, SPEC_AUGMENT,
    LR, WEIGHT_DECAY,
)
from binary_pla.data_loader import _load_cache, _transient_groups
from binary_pla.models import build_binary_model
from binary_pla.trainer import _run_epoch, _EarlyStopping
from binary_pla.augmentation import SpecAugmentDGT
from binary_pla.results_io import save_results

%matplotlib inline
plt.rcParams['figure.dpi'] = 110

N_CLASSES  = N_DEVICES
MODEL_NAME = 'bigru_dgt'
EPOCHS     = NB04_DL['bigru_dgt']['epochs']
PATIENCE   = NB04_DL['bigru_dgt']['patience']

augment = SpecAugmentDGT(**SPEC_AUGMENT)

print(f'Seeds      : {SEEDS}')
print(f'Model      : {MODEL_NAME}  →  {N_CLASSES}-class')
print(f'Augment    : {augment.__class__.__name__}')
print(f'Random base: {1/N_CLASSES:.3f}')

## 2. Data loading and split

In [2]:
X_all, y_all, _ = _load_cache('dgt', use_windowed=True)

print(f'X shape : {X_all.shape}   dtype={X_all.dtype}')
print(f'y shape : {y_all.shape}   unique labels={np.unique(y_all)}')
print()
print(f'Samples per device:')
for lbl in np.unique(y_all):
    print(f'  device{lbl+1} (label {lbl}): {(y_all==lbl).sum()}')

X shape : (1340, 150, 150)   dtype=float32
y shape : (1340,)   unique labels=[ 0  1  2  3  4  5  6  7  8  9 10 11]

Samples per device:
  device1 (label 0): 130
  device2 (label 1): 130
  device3 (label 2): 130
  device4 (label 3): 120
  device5 (label 4): 130
  device6 (label 5): 130
  device7 (label 6): 130
  device8 (label 7): 130
  device9 (label 8): 130
  device10 (label 9): 60
  device11 (label 10): 60
  device12 (label 11): 60


In [3]:
def make_split(X, y, seed, val_frac=VAL_FRAC, test_frac=TEST_FRAC, n_win=N_WIN):
    """
    Group-aware 70/15/15 split — all windows of one transient stay together.
    Normalization fitted on train only.
    Returns (X_tr, y_tr, X_va, y_va, X_te, y_te).
    """
    all_labels = list(range(N_CLASSES))
    groups = _transient_groups(y, all_labels, n_win)

    gss1 = GroupShuffleSplit(1, test_size=test_frac, random_state=seed)
    tv_idx, te_idx = next(gss1.split(X, y, groups=groups))
    X_tv, y_tv, g_tv = X[tv_idx], y[tv_idx], groups[tv_idx]
    X_te, y_te        = X[te_idx], y[te_idx]

    val_adj = val_frac / (1.0 - test_frac)
    gss2 = GroupShuffleSplit(1, test_size=val_adj, random_state=seed)
    tr_idx, va_idx = next(gss2.split(X_tv, y_tv, groups=g_tv))
    X_tr, y_tr = X_tv[tr_idx], y_tv[tr_idx]
    X_va, y_va = X_tv[va_idx], y_tv[va_idx]

    # Z-score fitted on train only
    flat = X_tr.reshape(len(X_tr), -1)
    mu, sig = flat.mean(0), flat.std(0) + 1e-8
    def _norm(X):
        f = X.reshape(len(X), -1)
        return ((f - mu) / sig).astype(np.float32).reshape(X.shape)

    return _norm(X_tr), y_tr, _norm(X_va), y_va, _norm(X_te), y_te


def make_loader(X, y, shuffle, batch_size=BATCH_SIZE):
    ds = TensorDataset(
        torch.from_numpy(np.ascontiguousarray(X)),
        torch.from_numpy(y.astype(np.int64)),
    )
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle)


def multiclass_weights(y_train):
    """Inverse-frequency weights for 12 classes."""
    counts = np.bincount(y_train, minlength=N_CLASSES).astype(float)
    w = 1.0 / np.maximum(counts, 1)
    return torch.tensor((w / w.sum() * N_CLASSES), dtype=torch.float32)


# Sanity check on seed=42
X_tr, y_tr, X_va, y_va, X_te, y_te = make_split(X_all, y_all, seed=42)
print(f'Split sizes (seed=42):')
print(f'  train: {len(X_tr)}  val: {len(X_va)}  test: {len(X_te)}')
print(f'  train class counts: {np.bincount(y_tr, minlength=N_CLASSES)}')

Split sizes (seed=42):
  train: 930  val: 200  test: 210
  train class counts: [ 80 100 100  60  90 100  70  70 110  50  50  50]


## 3. Multi-seed training

In [ ]:
seed_results = []

for seed in SEEDS:
    print(f'\n─── seed={seed} ─────────────────────────────')
    X_tr, y_tr, X_va, y_va, X_te, y_te = make_split(X_all, y_all, seed=seed)

    train_loader = make_loader(X_tr, y_tr, shuffle=True)
    val_loader   = make_loader(X_va, y_va, shuffle=False)
    test_loader  = make_loader(X_te, y_te, shuffle=False)

    torch.manual_seed(seed)
    model     = build_binary_model(MODEL_NAME, n_classes=N_CLASSES)
    weights   = multiclass_weights(y_tr)
    criterion = nn.CrossEntropyLoss(weight=weights)
    optimizer = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)
    stopper   = _EarlyStopping(PATIENCE)

    for epoch in range(1, EPOCHS + 1):
        tr_loss, tr_acc = _run_epoch(model, train_loader, criterion, optimizer, True,  augment)
        vl_loss, vl_acc = _run_epoch(model, val_loader,   criterion, optimizer, False, None)
        scheduler.step(vl_loss)
        if stopper.step(vl_loss, model):
            print(f'  early stop @ epoch {epoch}  val_acc={vl_acc:.3f}')
            break
    stopper.restore(model)

    # ── Evaluate on test ──────────────────────────────────────────────────────
    model.eval()
    all_preds, all_true = [], []
    with torch.no_grad():
        for X_b, y_b in test_loader:
            preds = model(X_b).argmax(1).numpy()
            all_preds.append(preds)
            all_true.append(y_b.numpy())
    all_preds = np.concatenate(all_preds)
    all_true  = np.concatenate(all_true)

    acc      = float(accuracy_score(all_true, all_preds))
    macro_f1 = float(f1_score(all_true, all_preds, average='macro',    zero_division=0))
    wgt_f1   = float(f1_score(all_true, all_preds, average='weighted', zero_division=0))
    cm       = confusion_matrix(all_true, all_preds, labels=list(range(N_CLASSES)))

    print(f'  accuracy={acc:.3f}  macro-F1={macro_f1:.3f}  weighted-F1={wgt_f1:.3f}')

    per_class_acc = cm.diagonal() / np.maximum(cm.sum(axis=1), 1)

    seed_results.append({
        'seed':         seed,
        'accuracy':     acc,
        'macro_f1':     macro_f1,
        'weighted_f1':  wgt_f1,
        'per_class_acc': per_class_acc.tolist(),
        'confusion_matrix': cm.tolist(),
    })

## 4. Results

In [ ]:
accs      = [r['accuracy']    for r in seed_results]
macro_f1s = [r['macro_f1']   for r in seed_results]
wgt_f1s   = [r['weighted_f1'] for r in seed_results]

print(f'\n{"─"*52}')
print(f'  BiGRU + SpecAugment — 12-class — {len(SEEDS)} seeds')
print(f'{"─"*52}')
print(f'  Accuracy     : {np.mean(accs):.3f} ± {np.std(accs):.3f}')
print(f'  Macro-F1     : {np.mean(macro_f1s):.3f} ± {np.std(macro_f1s):.3f}')
print(f'  Weighted-F1  : {np.mean(wgt_f1s):.3f} ± {np.std(wgt_f1s):.3f}')
print(f'  Random base  : {1/N_CLASSES:.3f}')
print(f'{"─"*52}')

print(f'\nPer-seed:')
print(f'  {"Seed":>6} {"Accuracy":>10} {"Macro-F1":>10} {"Wtd-F1":>10}')
print('  ' + '─'*38)
for r in seed_results:
    print(f'  {r["seed"]:>6} {r["accuracy"]:>10.3f} {r["macro_f1"]:>10.3f} {r["weighted_f1"]:>10.3f}')

In [ ]:
# Per-class accuracy bar chart
per_class = np.array([r['per_class_acc'] for r in seed_results])  # (n_seeds, 12)
means = per_class.mean(0)
stds  = per_class.std(0)

fig, ax = plt.subplots(figsize=(12, 4))
x = np.arange(N_CLASSES)
ax.bar(x, means, yerr=stds, capsize=4, color='steelblue', alpha=0.85)
ax.axhline(1/N_CLASSES, color='red', linestyle='--', lw=0.8, label=f'random ({1/N_CLASSES:.2f})')
ax.set_xticks(x)
ax.set_xticklabels(DEVICE_NAMES, rotation=30, fontsize=8)
ax.set_ylabel('Per-class accuracy (mean ± std across seeds)')
ax.set_ylim(0, 1.05)
ax.set_title('Per-device classification accuracy — BiGRU 12-class')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# Averaged confusion matrix (row-normalised)
avg_cm = np.array([r['confusion_matrix'] for r in seed_results]).mean(0)
norm_cm = avg_cm / np.maximum(avg_cm.sum(axis=1, keepdims=True), 1)

fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(norm_cm, vmin=0, vmax=1, cmap='Blues')
plt.colorbar(im, ax=ax, fraction=0.046, label='fraction correctly classified')
ax.set_xticks(range(N_CLASSES))
ax.set_yticks(range(N_CLASSES))
ax.set_xticklabels(DEVICE_NAMES, rotation=45, ha='right', fontsize=7)
ax.set_yticklabels(DEVICE_NAMES, fontsize=7)
ax.set_xlabel('Predicted device')
ax.set_ylabel('True device')
ax.set_title('Confusion matrix — BiGRU 12-class (averaged over seeds, row-normalised)')
for i in range(N_CLASSES):
    for j in range(N_CLASSES):
        v = norm_cm[i, j]
        ax.text(j, i, f'{v:.2f}', ha='center', va='center',
                fontsize=5, color='white' if v > 0.5 else 'black')
plt.tight_layout()
plt.show()

## 4b. MLP-FV — multi-class (preprocessed features)

MLP operating on the 505-dim RF-DNA feature vectors from `fv_windowed.h5`.
No augmentation (SpecAugmentDGT is specific to 2-D DGT matrices).
Same seeds and split logic as the BiGRU section — directly comparable.

In [4]:
# Load FV windowed cache (same labels, same transient ordering)
X_fv, y_fv, _ = _load_cache('fv', use_windowed=True)
print(f'FV cache: X={X_fv.shape}  y={y_fv.shape}')

FV cache: X=(1340, 505)  y=(1340,)


In [ ]:
seed_results_mlp = []

for seed in SEEDS:
    print(f'\n─── seed={seed} (MLP-FV) ────────────────────────')
    X_tr, y_tr, X_va, y_va, X_te, y_te = make_split(X_fv, y_fv, seed=seed)

    train_loader = make_loader(X_tr, y_tr, shuffle=True)
    val_loader   = make_loader(X_va, y_va, shuffle=False)
    test_loader  = make_loader(X_te, y_te, shuffle=False)

    torch.manual_seed(seed)
    model     = build_binary_model('mlp_fv', n_classes=N_CLASSES)
    weights   = multiclass_weights(y_tr)
    criterion = nn.CrossEntropyLoss(weight=weights)
    optimizer = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)
    stopper   = _EarlyStopping(NB04_DL['mlp_fv']['patience'])

    for epoch in range(1, NB04_DL['mlp_fv']['epochs'] + 1):
        _run_epoch(model, train_loader, criterion, optimizer, True,  None)
        vl_loss, vl_acc = _run_epoch(model, val_loader, criterion, optimizer, False, None)
        scheduler.step(vl_loss)
        if stopper.step(vl_loss, model):
            print(f'  early stop @ epoch {epoch}  val_acc={vl_acc:.3f}')
            break
    stopper.restore(model)

    model.eval()
    all_preds, all_true = [], []
    with torch.no_grad():
        for X_b, y_b in test_loader:
            all_preds.append(model(X_b).argmax(1).numpy())
            all_true.append(y_b.numpy())
    all_preds = np.concatenate(all_preds)
    all_true  = np.concatenate(all_true)

    acc      = float(accuracy_score(all_true, all_preds))
    macro_f1 = float(f1_score(all_true, all_preds, average='macro',    zero_division=0))
    wgt_f1   = float(f1_score(all_true, all_preds, average='weighted', zero_division=0))
    cm       = confusion_matrix(all_true, all_preds, labels=list(range(N_CLASSES)))
    per_class_acc = cm.diagonal() / np.maximum(cm.sum(axis=1), 1)

    print(f'  accuracy={acc:.3f}  macro-F1={macro_f1:.3f}  weighted-F1={wgt_f1:.3f}')
    seed_results_mlp.append({
        'seed': seed, 'accuracy': acc, 'macro_f1': macro_f1,
        'weighted_f1': wgt_f1, 'per_class_acc': per_class_acc.tolist(),
        'confusion_matrix': cm.tolist(),
    })

In [6]:
accs_mlp  = [r['accuracy']    for r in seed_results_mlp]
mf1_mlp   = [r['macro_f1']   for r in seed_results_mlp]
wf1_mlp   = [r['weighted_f1'] for r in seed_results_mlp]

print(f'\n{"─"*60}')
print(f'  Model comparison — 12-class  ({len(SEEDS)} seeds)')
print(f'{"─"*60}')
print(f'  {"Model":<20} {"Accuracy":>12} {"Macro-F1":>12} {"Wtd-F1":>12}')
print(f'  {"─"*56}')
print(f'  {"BiGRU+SpecAug (DGT)":<20} '
      f'{np.mean(accs):.3f}±{np.std(accs):.3f}  '
      f'{np.mean(macro_f1s):.3f}±{np.std(macro_f1s):.3f}  '
      f'{np.mean(wgt_f1s):.3f}±{np.std(wgt_f1s):.3f}')
print(f'  {"MLP-FV (features)":<20} '
      f'{np.mean(accs_mlp):.3f}±{np.std(accs_mlp):.3f}  '
      f'{np.mean(mf1_mlp):.3f}±{np.std(mf1_mlp):.3f}  '
      f'{np.mean(wf1_mlp):.3f}±{np.std(wf1_mlp):.3f}')
print(f'  {"Random baseline":<20} {1/N_CLASSES:.3f}{"":>12}{"":>12}')
print(f'{"─"*60}')

# Per-class comparison bar chart
pc_bigru = np.array([r['per_class_acc'] for r in seed_results]).mean(0)
pc_mlp   = np.array([r['per_class_acc'] for r in seed_results_mlp]).mean(0)
x = np.arange(N_CLASSES)
w = 0.35

fig, ax = plt.subplots(figsize=(13, 4))
ax.bar(x - w/2, pc_bigru, w, label='BiGRU+SpecAug (DGT)', color='steelblue', alpha=0.85)
ax.bar(x + w/2, pc_mlp,   w, label='MLP-FV (features)',   color='darkorange', alpha=0.85)
ax.axhline(1/N_CLASSES, color='red', linestyle='--', lw=0.8, label=f'random ({1/N_CLASSES:.2f})')
ax.set_xticks(x)
ax.set_xticklabels(DEVICE_NAMES, rotation=30, fontsize=8)
ax.set_ylabel('Per-class accuracy (mean across seeds)')
ax.set_ylim(0, 1.05)
ax.set_title('Per-device accuracy — BiGRU vs MLP-FV (12-class)')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()


────────────────────────────────────────────────────────────
  Model comparison — 12-class  (5 seeds)
────────────────────────────────────────────────────────────
  Model                    Accuracy     Macro-F1       Wtd-F1
  ────────────────────────────────────────────────────────


NameError: name 'accs' is not defined

## 5. Connection to PLA narrative

Multi-class classification uses the full dataset (all 12 devices, all windowed transients)
while binary PLA trains one classifier per authorized device on a subset.

If multi-class accuracy is meaningfully higher than binary LOTO ADR, the gap confirms
that **per-classifier sample starvation**, not model capacity, is the binding constraint
in the binary PLA framework.

In [ ]:
binary_loto_adr  = 0.627   # BiGRU LOTO mean ADR from notebook 02 (threshold=0.5)
bigru_mc_acc     = float(np.mean(accs))
mlp_mc_acc       = float(np.mean(accs_mlp))
best_mc_acc      = max(bigru_mc_acc, mlp_mc_acc)
best_model       = 'BiGRU+SpecAug' if bigru_mc_acc >= mlp_mc_acc else 'MLP-FV'
gap              = best_mc_acc - binary_loto_adr

print(f'{"─"*64}')
print(f'  Comparison: multi-class vs binary PLA')
print(f'{"─"*64}')
print(f'  {"Setting":<38} {"Metric":>10} {"Value":>10}')
print(f'  {"─"*60}')
print(f'  {"Binary PLA — BiGRU LOTO":<38} {"mean ADR":>10} {binary_loto_adr:>10.3f}')
print(f'  {"Multi-class — BiGRU+SpecAug (DGT)":<38} {"accuracy":>10} {bigru_mc_acc:>10.3f}')
print(f'  {"Multi-class — MLP-FV (features)":<38} {"accuracy":>10} {mlp_mc_acc:>10.3f}')
print(f'  {"Random baseline":<38} {"accuracy":>10} {1/N_CLASSES:>10.3f}')
print(f'  {"─"*60}')
print(f'  Best multi-class: {best_model}  gap vs binary LOTO: {gap:+.3f}')
print()
if gap > 0.05:
    print('  → Multi-class is substantially higher: per-classifier sample')
    print('    starvation is the primary constraint in binary PLA.')
elif gap > 0:
    print('  → Multi-class is marginally higher: both dataset size and')
    print('    the one-vs-rest framing contribute to the gap.')
else:
    print('  → Models are comparable: the constraint is the total dataset')
    print('    size, independent of the binary framing.')

## 6. Save results

In [ ]:
from binary_pla.config import NB04_ID
from binary_pla.results_io import save_results

def _seed_to_dict(r):
    return {
        'seed':          r['seed'],
        'accuracy':      float(r['accuracy']),
        'macro_f1':      float(r['macro_f1']),
        'weighted_f1':   float(r['weighted_f1']),
        'per_class_acc': [float(v) for v in r['per_class_acc']],
    }

def _summary(results):
    a = [r['accuracy']    for r in results]
    m = [r['macro_f1']   for r in results]
    w = [r['weighted_f1'] for r in results]
    return {
        'accuracy_mean':    float(np.mean(a)), 'accuracy_std':    float(np.std(a)),
        'macro_f1_mean':    float(np.mean(m)), 'macro_f1_std':    float(np.std(m)),
        'weighted_f1_mean': float(np.mean(w)), 'weighted_f1_std': float(np.std(w)),
        'random_baseline':  round(1 / N_CLASSES, 4),
    }

results_payload = {
    'bigru_dgt': {
        'per_seed': [_seed_to_dict(r) for r in seed_results],
        '_summary': _summary(seed_results),
    },
    'mlp_fv': {
        'per_seed': [_seed_to_dict(r) for r in seed_results_mlp],
        '_summary': _summary(seed_results_mlp),
    },
}

save_results(
    NB04_ID, 'all_devices', results_payload,
    extra_meta={
        'n_classes': N_CLASSES,
        'seeds':     list(SEEDS),
        'epochs':    EPOCHS,
        'patience':  PATIENCE,
        'augment_bigru': f'SpecAugmentDGT({SPEC_AUGMENT})',
        'augment_mlp':   'none',
        'split': f'70/15/15 group-aware (n_win={N_WIN})',
    },
)

## 7. Export figures and numbers for report

In [ ]:
import os, json
import numpy as np
import matplotlib.pyplot as plt

# ── Paths ─────────────────────────────────────────────────────────────────────
FIG_DIR = os.path.join(ROOT, 'report', 'figures')
RES_DIR = os.path.join(ROOT, 'report', 'results')
os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(RES_DIR, exist_ok=True)

def _savefig(name):
    path = os.path.join(FIG_DIR, name)
    plt.savefig(path, bbox_inches='tight', dpi=150)
    plt.close()
    print(f'  saved → {name}')

# ── Load numbers from saved JSON ───────────────────────────────────────────────
with open(os.path.join(ROOT, 'results', 'nb04_multiclass_all_devices.json')) as f:
    _nb04 = json.load(f)
_r4 = _nb04.get('results', _nb04)
_bg = _r4['bigru_dgt']
_bg_seeds = _bg['per_seed']
_bg_acc  = np.mean([s['accuracy']  for s in _bg_seeds])
_bg_std  = np.std( [s['accuracy']  for s in _bg_seeds])
_bg_f1   = np.mean([s['macro_f1']  for s in _bg_seeds])
_bg_f1s  = np.std( [s['macro_f1']  for s in _bg_seeds])

# Binary LOTO ADR reference (from nb02)
_BINARY_LOTO_ADR = 0.676   # mean_adr, bigru_dgt LOTO, trial_1

print(f'BiGRU 12-class: acc={_bg_acc:.3f}±{_bg_std:.3f}  macro-F1={_bg_f1:.3f}±{_bg_f1s:.3f}')
print(f'Binary LOTO ADR (reference): {_BINARY_LOTO_ADR:.3f}')

# ── Figure 1: Confusion matrix (requires in-memory seed_results) ──────────────
try:
    avg_cm = np.array([r['confusion_matrix'] for r in seed_results]).mean(0)
    norm_cm = avg_cm / np.maximum(avg_cm.sum(axis=1, keepdims=True), 1)

    fig, ax = plt.subplots(figsize=(9, 7))
    im = ax.imshow(norm_cm, vmin=0, vmax=1, cmap='Blues')
    plt.colorbar(im, ax=ax, fraction=0.046, label='fraction correctly classified')
    ax.set_xticks(range(N_CLASSES))
    ax.set_yticks(range(N_CLASSES))
    ax.set_xticklabels(DEVICE_NAMES, rotation=45, ha='right', fontsize=7)
    ax.set_yticklabels(DEVICE_NAMES, fontsize=7)
    ax.set_xlabel('Predicted device')
    ax.set_ylabel('True device')
    ax.set_title('Confusion matrix — BiGRU 12-class (averaged over seeds, row-normalised)')
    for i in range(N_CLASSES):
        for j in range(N_CLASSES):
            v = norm_cm[i, j]
            ax.text(j, i, f'{v:.2f}', ha='center', va='center',
                    fontsize=5, color='white' if v > 0.5 else 'black')
    plt.tight_layout()
    _savefig('fig_mc_confusion.pdf')
except NameError:
    print('  [skip] seed_results not in memory — re-run mc-train cell first')

# ── Figure 2: Per-device accuracy bar chart ────────────────────────────────────
try:
    per_class = np.array([r['per_class_acc'] for r in seed_results])
    means = per_class.mean(0)
    stds  = per_class.std(0)

    fig, ax = plt.subplots(figsize=(12, 4))
    x = np.arange(N_CLASSES)
    ax.bar(x, means, yerr=stds, capsize=4, color='steelblue', alpha=0.85)
    ax.axhline(1/N_CLASSES, color='red', linestyle='--', lw=0.8, label=f'random ({1/N_CLASSES:.2f})')
    ax.set_xticks(x)
    ax.set_xticklabels(DEVICE_NAMES, rotation=30, fontsize=8)
    ax.set_ylabel('Per-class accuracy (mean ± std across seeds)')
    ax.set_ylim(0, 1.05)
    ax.set_title('Per-device classification accuracy — BiGRU 12-class')
    ax.legend(fontsize=9)
    plt.tight_layout()
    _savefig('fig_mc_perclass.pdf')
except NameError:
    print('  [skip] seed_results not in memory — re-run mc-train cell first')

# ── Figure 3: Multi-class vs binary LOTO bar chart ────────────────────────────
_labels  = ['Binary LOTO\n(BiGRU)', 'Multi-class\n(BiGRU)', 'Random\nbaseline']
_values  = [_BINARY_LOTO_ADR, _bg_acc, 1/N_CLASSES]
_errors  = [0, _bg_std, 0]
_colors  = ['steelblue', 'seagreen', 'lightgray']

fig, ax = plt.subplots(figsize=(5, 4))
bars = ax.bar(_labels, _values, yerr=_errors, capsize=5,
              color=_colors, alpha=0.85, width=0.5)
ax.axhline(0.5, color='gray', linestyle='--', lw=0.8, label='chance (0.5)')
ax.set_ylim(0, 1.0)
ax.set_ylabel('ADR / Accuracy')
ax.set_title('Multi-class vs Binary LOTO — BiGRU')
ax.legend(fontsize=9)
for bar, val in zip(bars, _values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f'{val:.3f}', ha='center', fontsize=9)
plt.tight_layout()
_savefig('fig_mc_comparison.pdf')

# ── Write numbers to report/results/numbers.tex (append mc block) ─────────────
_numbers_path = os.path.join(RES_DIR, 'numbers.tex')
_mc_block = (
    f'% ── Multi-class probe (nb04) ────────────────────────────────────────────\n'
    f'\\newcommand{{\\mcAccuracy}}{{${_bg_acc:.3f} \\pm {_bg_std:.3f}$}}\n'
    f'\\newcommand{{\\mcMacroFone}}{{${_bg_f1:.3f} \\pm {_bg_f1s:.3f}$}}\n'
    f'\\newcommand{{\\mcRandomBase}}{{{1/N_CLASSES:.3f}}}\n'
    f'\\newcommand{{\\mcBinaryLOTO}}{{{_BINARY_LOTO_ADR:.3f}}}\n'
)

# Append if file exists (nb03 already wrote SVM/DL/LOTO numbers), else create
_mode = 'a' if os.path.exists(_numbers_path) else 'w'
with open(_numbers_path, _mode) as f:
    f.write('\n' + _mc_block)
print(f'  wrote mc numbers → report/results/numbers.tex ({_mode})')
print('\nDone. Figures saved to report/figures/')
